# Igniters Week 8 — Exercise Solutions

Solutions for **Igniters_week8_exercises.ipynb**. Run code that depends on `agents`, `deal_agent_framework`, or `hello` from the **week8** directory (e.g. open in Jupyter with week8 as cwd), or uncomment the path setup in a cell where needed.

---
## Exercise 1 (Day 1) — Modal and hello_europe

**Task:** Run `hello_europe` and state what Modal is used for in this project.

**Solution:** In `../day1.ipynb` (or from the week8 folder), after importing the hello app:

```python
from hello import app, hello, hello_europe

with app.run():
    reply = hello_europe.local()
print(reply)
```

**One-sentence answer:** Modal is used in this project to deploy the **specialist pricer** (fine-tuned or ML model) as a serverless function so the deal-hunting pipeline can call it remotely for price estimates.

In [ ]:
# Run from week8 directory so 'hello' is importable
# from hello import app, hello_europe
# with app.run():
#     reply = hello_europe.local()
# print(reply)

---
## Exercise 2 (Day 2) — RAG query and visualization

**Task:** Change MAXIMUM_DATAPOINTS or run a single RAG query (Chroma + LLM).

**Solution (A) — Smaller visualization:** In `../day2.ipynb`, set `MAXIMUM_DATAPOINTS = 2_000` (or 5_000) before the cells that do `collection.get(..., limit=MAXIMUM_DATAPOINTS)` and run the t-SNE and plotly cells. Fewer points = faster and lighter.

**Solution (B) — Single RAG query:** After running the day2 setup (encoder, collection, train, test, find_similars, make_context, messages_for, and completion), run the following for one test item (e.g. test[0]): retrieve top-k similar docs/prices, build context, call the LLM, and print the model’s estimated price and the top-3 retrieved prices.

In [ ]:
# Run this in ../day2.ipynb after encoder, collection, test, find_similars, make_context, messages_for exist.
# from litellm import completion

# item = test[0]
# documents, prices = find_similars(item)
# print("Top-3 retrieved prices:", prices[:3])
# response = completion(model="gpt-5.1", messages=messages_for(item, documents, prices), reasoning_effort="none", seed=42)
# est = response.choices[0].message.content
# print("LLM estimated price:", est)
# print("Actual price:", item.price)

---
## Exercise 3 (Day 3) — Request 3 deals instead of 5

**Task:** Change the scanner prompt so the LLM returns exactly 3 deals.

**Solution:** In `../day3.ipynb`, update the system and user prompts to say "3" instead of "5":

- In **SYSTEM_PROMPT**: change "5 most detailed deals" and "exactly 5 deals" to "3 most detailed deals" and "exactly 3 deals".
- In **USER_PROMPT_PREFIX**: change "5 deals" to "3 deals".
- In **USER_PROMPT_SUFFIX**: use `"\n\nInclude exactly 3 deals, no more."`

Re-run the cell that builds the user prompt and calls the LLM; parse the JSON response and confirm it has exactly 3 entries.

---
## Exercise 4 (Day 4) — Add a fourth tool: log_deal_to_file

**Task:** Add `log_deal_to_file(description, price)` and register it with the planner.

**Solution:** (1) Implement the function and (2) add its JSON schema to the tools list. The planner (e.g. in `agents/planning_agent.py`) must receive this tool in its list of available tools; if you are defining tools in the day4 notebook and passing them to an agent, append the new tool there.

In [ ]:
LOG_FILE = "deals_log.txt"

def log_deal_to_file(description: str, price: float) -> str:
    """Append one line with description and price to a log file."""
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(f"{description}\t{price}\n")
    return f"Logged to {LOG_FILE}"

log_function = {
    "name": "log_deal_to_file",
    "description": "Append a deal (description and price) to a log file for later review",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {"type": "string", "description": "Short description of the deal"},
            "price": {"type": "number", "description": "Price of the deal"}
        },
        "required": ["description", "price"],
        "additionalProperties": False
    }
}

# Add log_function to the list of tools you pass to the planning agent (e.g. tools = [scan_function, estimate_function, notify_function, log_function]).

---
## Exercise 5 (Day 5) — "Refresh deals" button

**Task:** Add a button that runs the agent and updates the opportunities table.

**Solution:** Add a **Refresh deals** button that calls `agent_framework.run()`, then returns the updated `agent_framework.memory` to both the `gr.State` (opportunities) and the Dataframe so the table refreshes. Example pattern (integrate into the existing Gradio Blocks in `../day5.ipynb`):

In [ ]:
# Run from week8 so deal_agent_framework and agents are importable.
# Add this inside your existing gr.Blocks():

# def refresh_deals(opps):
#     agent_framework.run()  # runs planner, appends to memory, writes to disk
#     new_opps = agent_framework.memory
#     return new_opps, get_table(new_opps)

# refresh_btn = gr.Button("Refresh deals")
# refresh_btn.click(
#     refresh_deals,
#     inputs=[opportunities],
#     outputs=[opportunities, opportunities_dataframe]
# )

# So: one output is the new State value, the other is the table data for the Dataframe.

---
**Summary**

| Exercise | Day | Summary |
|----------|-----|--------|
| 1 | 1 | Run `hello_europe.local()`; Modal deploys the specialist pricer. |
| 2 | 2 | Set MAXIMUM_DATAPOINTS or run find_similars + completion for one item. |
| 3 | 3 | Change prompts from "5" to "3" deals and re-run LLM. |
| 4 | 4 | Add `log_deal_to_file` + JSON schema and register with planner. |
| 5 | 5 | Button that calls `agent_framework.run()` and updates State + Dataframe. |